In [21]:
import scipy.stats
from openai import OpenAI
from math import exp
import numpy as np
import json
from scipy.stats import entropy
import math
from IPython.display import display, HTML
import os
import ast
np.set_printoptions(legacy='1.25')
import pprint


In [22]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


#### GPT methods

In [23]:
def get_completion(
    messages: list[dict[str, str]],
    model: str = "gpt-4",
    max_tokens=500,
    temperature=0,
    stop=None,
    seed=123,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stop": stop,
        "seed": seed,
        "logprobs": logprobs,
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.chat.completions.create(**params)
    return completion

In [24]:
def get_ans_token_logprobs(response, anskey):
    tokenprobs = response.choices[0].logprobs.content
    i = 0
    token = tokenprobs[i].token.strip()
    #print(i, token)
    while token != anskey:
        i += 1
        token = tokenprobs[i].token.strip()
        #print('forward {}, {}'.format(i, token))

    while token != '{' and token != '{\'' and token != '{\"':
        i -= 1
        token = tokenprobs[i].token.strip()
        #print('backward {}, {}'.format(i, token))

    json_begin = i
    #print('json_begins at {}'.format(json_begin))

    while token != '}' and token != '}\'' and token != '}.' and token != '\'}' and token != '}\"':
        i += 1
        token = tokenprobs[i].token.strip()
        #print('forward {}, {}'.format(i, token))

    json_end = i
    #print(json_begin, json_end)
    response_list = tokenprobs[json_begin: json_end+1]
    response_string = "".join([o.token for o in response_list]).strip('.')
    response_dict = ast.literal_eval(response_string)

    ans_logprobs = None

    #print(str(response_dict[anskey]))
    for item in response_list:
        if item.token == str(response_dict[anskey]):
            ans_logprobs = item.top_logprobs


    return response_dict, ans_logprobs



In [25]:
def get_answer(question, prompt, anskey):
    num_next_tokens = 5
    prompt_up = prompt.format(question=question)
    #print(prompt_up)
    API_RESPONSE = get_completion(
        [{"role": "user", "content": prompt_up}],
        model="gpt-4",
        logprobs=True,
        top_logprobs=num_next_tokens,
    )

    response_dict, ans_logprobs = get_ans_token_logprobs(API_RESPONSE, anskey)

    alters = []
    probs = []
    for item in ans_logprobs:
        linear_prob = np.round(np.exp(item.logprob)*100, 3)
        alters.append((item.token, linear_prob, item.logprob))
        probs.append(linear_prob)


    return_dict = {
        '1. prompt': prompt_up,
        '2. response': API_RESPONSE.choices[0].message.content,
        '3. response_dict': response_dict,
        '4. entropy': np.round(scipy.stats.entropy(probs, base=2)/scipy.stats.entropy([0.2, 0.2, 0.2, 0.2, 0.2], base=2), 3),
        '5. alternatives': alters
    }

    return return_dict


#### Prompts

In [6]:
basePROMPT = """You will be given a math word problem. Provide a numeric answer to the problem. MAKE SURE to only provide a numeric answer. Your final response should be a dictionary with key 'ans'.
Problem: {question}"""

In [7]:
confidencePROMPT = """You will be given a math word problem. Provide a numeric answer to the problem. MAKE SURE to only provide a numeric answer. Your confidence is how much you trust your answer to be correct and is on a scale of 1-10. Your final response should be a dictionary with key 'ans' and 'confidence'.
Problem: {question}"""

In [8]:
emptyPROMPT="""Your final response should be a dictionary with key 'ans'. Problem: {question}"""

In [9]:
cotPROMPT=""""You will be given a math word problem. Provide a numeric answer to the problem. Think step by step. Your final response should be a dictionary with key 'ans'.
Problem: {question}"""

#### Problem text

LLMs, particularly those with reasoning, are known to be good at solving algebra word problems. Here, I experiment with various levels of ambiguity in the question. The following problems are variations of the same type. The first two problems have an unambiguous answer that can be calculated in a straight forward fashion. The third problem doesn't explicitly specify which fight the question is about. The fourth problem requires additional layers of reasoning.

In [10]:
problems = {
            "straight_opt1_01": {
                    "question": "On average Joe throws 25 punches per minute. First fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds of 5 minutes. How many punches did he throw in the first fight?",
                    "correct_ans": 375,
                    "category": "unambiguous"
                },
            "straight_opt2_01":{
                    "question": "On average Joe throws 25 punches per minute. First fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds of 5 minutes. How many punches did he throw in the second fight?",
                    "correct_ans": 500,
                    "category": "unambiguous"
                },
            "ambiguous_01": {
                    "question": "On average Joe throws 25 punches per minute. First fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds of 5 minutes. How many punches did he throw in the fight?",
                    "correct_ans": None,
                    "category": "ambiguous"
                },
            "total_01": {
                    "question": "On average Joe throws 25 punches per minute. First fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds of 5 minutes. How many punches did he throw in both fights?",
                    "correct_ans": 875,
                    "category": "unambiguous"
                }
}

#### Observations

We begin with the easy, unambigous questions.


In [11]:
ans = get_answer(question=problems['straight_opt1_01']['question'], prompt=basePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: On average Joe throws 25 punches per minute. First '
              'fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds '
              'of 5 minutes. How many punches did he throw in the first fight?',
 '2. response': "{'ans': 375}",
 '3. response_dict': {'ans': 375},
 '4. entropy': 0.0,
 '5. alternatives': [('375', 100.0, 0.0),
                     ('325', 0.0, -18.034464),
                     ('25', 0.0, -18.438463),
                     ('75', 0.0, -19.007116),
                     (' ', 0.0, -19.025997)]}


In [12]:
ans = get_answer(question=problems['straight_opt2_01']['question'], prompt=basePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: On average Joe throws 25 punches per minute. First '
              'fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds '
              'of 5 minutes. How many punches did he throw in the second '
              'fight?',
 '2. response': "{'ans': 500}",
 '3. response_dict': {'ans': 500},
 '4. entropy': 0.0,
 '5. alternatives': [('500', 99.998, -1.831257e-05),
                     ('100', 0.001, -11.603904),
                     ('200', 0.0, -12.443856),
                     ('600', 0.0, -13.573398),
                     ('125', 0.0, -13.640393)]}


For both unambiguous questions, regular prompting can generate correct answers. I report entropy of the answer which captures the degree to which the LLM is 'confused' about the answer. In both these cases, there is no confusion and hence, the entropy is 0. All the other alternatives considered by the LLM have very low probabilities. This is what we expect when reasoning works as expected.

Next, we consider the ambiguous question where the fight under consideration is not specified. We expect the entropy of the answer to be high, reflecting the LLMs is confused about the answer.

In [13]:
ans = get_answer(question=problems['ambiguous_01']['question'], prompt=basePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: On average Joe throws 25 punches per minute. First '
              'fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds '
              'of 5 minutes. How many punches did he throw in the fight?',
 '2. response': "{'ans': 675}",
 '3. response_dict': {'ans': 675},
 '4. entropy': 0.775,
 '5. alternatives': [('675', 38.839, -0.94574225),
                     ('105', 18.935, -1.6641833),
                     ('975', 11.492, -2.1635613),
                     ('110', 3.426, -3.3737226),
                     ('950', 3.171, -3.451239)]}


As expected, the entropy of the answer in very high reflecting the confusion within LLM's inference. However, the LLM does produce a response (675, which is incorrect). This behavior is very different from how a human would respond. Because the question is ambiguous, a human would ask 'which fight are you talking about?'. The secondary question elicits further information from the speaker and constrains ambiguity. LLMs cannot modulate their response based on 'confusion' in their inference systems.

Next, I investigate if the LLM is aware of its own confusion. I asked it to report its own confidence on the answer on a scale of 1-10.

In [14]:
ans = get_answer(question=problems['ambiguous_01']['question'], prompt=confidencePROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              'Your confidence is how much you trust your answer to be correct '
              'and is on a scale of 1-10. Your final response should be a '
              "dictionary with key 'ans' and 'confidence'.\n"
              'Problem: On average Joe throws 25 punches per minute. First '
              'fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds '
              'of 5 minutes. How many punches did he throw in the fight?',
 '2. response': "{'ans': 675, 'confidence': 10}",
 '3. response_dict': {'ans': 675, 'confidence': 10},
 '4. entropy': 0.836,
 '5. alternatives': [('675', 34.825, -1.0548251),
                     ('105', 19.296, -1.6452849),
                     ('975', 15.072, -1.89234),
                     ('112', 5.089, -2.9781382),
                     ('110', 3.69, -3.2996027)]}


In its response, the LLM claims to be highly confident (with a score of 10). It reports that it is highly confident of the answer. However, not only is the answer wrong but high answer entropy suggests that the LLM is not confident in the answer.

In [15]:
ans = get_answer(question=problems['total_01']['question'], prompt=basePROMPT, anskey='ans')
problems['total_01']
pprint.pprint(ans)

{'1. prompt': 'You will be given a math word problem. Provide a numeric answer '
              'to the problem. MAKE SURE to only provide a numeric answer. '
              "Your final response should be a dictionary with key 'ans'.\n"
              'Problem: On average Joe throws 25 punches per minute. First '
              'fight lasts 5 rounds of 3 minutes. Second fight lasts 4 rounds '
              'of 5 minutes. How many punches did he throw in both fights?',
 '2. response': '{"ans": 675}',
 '3. response_dict': {'ans': 675},
 '4. entropy': 0.319,
 '5. alternatives': [('675', 81.589, -0.20347306),
                     ('975', 4.246, -3.1591253),
                     ('105', 3.245, -3.4281225),
                     ('725', 1.932, -3.9465132),
                     ('825', 1.386, -4.2786093)]}


#### Prompts for common sense reasoning

In [26]:
baseCommonPROMPT = """You will be given a commonsense reasoning problem. You will then be provided possible answer options labeled with letters following the question mark. Provide the letter assigned to the correct answer to the problem. MAKE SURE to only provide a letter as an answer. Your final response should be a dictionary with key 'ans'.
Problem: {question}"""

In [40]:
multipleAnswerPROMPT =  """You will be given a commonsense reasoning problem. You will then be provided possible answer options labeled with letters following the question mark. Provide the letter assigned to the correct answer to the problem. There may be multiple answers. Provide a reasoning for why you made the choice. MAKE SURE to only provide one letter as an answer. Your final response should be a dictionary with key 'ans'.
Problem: {question}"""

In [60]:
multipleChoicesPROMPT =  """You will be given a commonsense reasoning problem. You will then be provided possible answer options labeled with letters following the question mark. Provide the letter assigned to the correct answer to the problem. There may be multiple answers. Provide a reasoning for why you made the choices including probabilities for the remaining probable answers. MAKE SURE to only provide one letter as an answer. Your final response should be a dictionary with key 'ans'.
Problem: {question}"""

#### Problem text

In [58]:
problems_common = {
            "straight_opt1_01": {
                    "question": "Sammy wanted to go to where the people were.  Where might he go? A: Race track, B: Populated areas, C: the desert, D: apartment, E: roadblock ",
                    "correct_ans": "B",
                    "category": "unambiguous"
                },

            "ambiguous_01": {
                    "question": "Sammy wanted to go to where the people were.  Where might he go? A: Restaurant, B: New York, C: the desert, D: gym in the morning, E: roadblock ",
                    "correct_ans": None,
                    "category": "ambiguous"
                },
            "ambiguous_02": {
                    "question": "Why would you be able to see through a door? A: Door is open, B: Door is made of glass, C: xray vision, D: door is made of wood, E: Door is made of plastic ",
                    "correct_ans": None,
                    "category": "ambiguous"
            },
            "ambiguous_03": {
                    "question": "Where can you go to have a person assist you in mailing a package? A: USPS, B: Fedex, C: UPS",
                    "correct_ans": None,
                    "category": "ambiguous"
            }
}

### Observations


In [19]:
ans = get_answer(question=problems_common['straight_opt1_01']['question'], prompt=baseCommonPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. MAKE SURE to only provide a '
              'letter as an answer. Your final response should be a dictionary '
              "with key 'ans'.\n"
              'Problem: Sammy wanted to go to where the people were.  Where '
              'might he go? A: Race track, B: Populated areas, C: the desert, '
              'D: apartment, E: roadblock ',
 '2. response': "{'ans': 'B'}",
 '3. response_dict': {'ans': 'B'},
 '4. entropy': 0.0,
 '5. alternatives': [('B', 100.0, -1.9361265e-07),
                     ('A', 0.0, -16.041931),
                     ('D', 0.0, -17.027851),
                     (' B', 0.0, -19.160372),
                     ('E', 0.0, -20.221634)]}


In [32]:
ans = get_answer(question=problems_common['ambiguous_01']['question'], prompt=baseCommonPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. MAKE SURE to only provide a '
              'letter as an answer. Your final response should be a dictionary '
              "with key 'ans'.\n"
              'Problem: Sammy wanted to go to where the people were.  Where '
              'might he go? A: Restaurant, B: New York, C: the desert, D: gym '
              'in the morning, E: roadblock ',
 '2. response': "{'ans': 'A'}",
 '3. response_dict': {'ans': 'A'},
 '4. entropy': 0.0,
 '5. alternatives': [('A', 99.997, -2.8444882e-05),
                     ('D', 0.001, -11.132879),
                     ('B', 0.001, -11.193483),
                     (' A', 0.0, -17.485836),
                     ('E', 0.0, -17.773832)]}


In [61]:
ans = get_answer(question=problems_common['ambiguous_01']['question'], prompt=multipleChoicesPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. There may be multiple answers. '
              'Provide a reasoning for why you made the choices including '
              'probabilities for the remaining probable answers. MAKE SURE to '
              'only provide one letter as an answer. Your final response '
              "should be a dictionary with key 'ans'.\n"
              'Problem: Sammy wanted to go to where the people were.  Where '
              'might he go? A: Restaurant, B: New York, C: the desert, D: gym '
              'in the morning, E: roadblock ',
 '2. response': "{'ans': 'A'}\n"
                '\n'
                'Reasoning: The question asks where Sammy might go if he '
                'wanted to be where the people were. The most 

In [37]:
ans = get_answer(question=problems_common['ambiguous_02']['question'], prompt=baseCommonPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. MAKE SURE to only provide a '
              'letter as an answer. Your final response should be a dictionary '
              "with key 'ans'.\n"
              'Problem: Why would you be able to see through a door? A: Door '
              'is open, B: Door is made of glass, C: xray vision, D: door is '
              'made of wood, E: Door is made of plastic ',
 '2. response': "{'ans': 'B'}",
 '3. response_dict': {'ans': 'B'},
 '4. entropy': 0.0,
 '5. alternatives': [('B', 100.0, -7.89631e-07),
                     ('A', 0.0, -14.093112),
                     (' B', 0.0, -17.349575),
                     ('C', 0.0, -18.364323),
                     ('D', 0.0, -18.935795)]}


In [41]:
ans = get_answer(question=problems_common['ambiguous_02']['question'], prompt=multipleAnswerPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. There may be multiple answers. '
              'Provide a reasoning for why you made the choice. MAKE SURE to '
              'only provide one letter as an answer. Your final response '
              "should be a dictionary with key 'ans'.\n"
              'Problem: Why would you be able to see through a door? A: Door '
              'is open, B: Door is made of glass, C: xray vision, D: door is '
              'made of wood, E: Door is made of plastic ',
 '2. response': "{'ans': 'B'} Reasoning: A door made of glass would allow you "
                'to see through it. While an open door or xray vision could '
                'technically allow you to see through a door, the question '
                'seem

In [47]:
ans = get_answer(question=problems_common['ambiguous_03']['question'], prompt=baseCommonPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. MAKE SURE to only provide a '
              'letter as an answer. Your final response should be a dictionary '
              "with key 'ans'.\n"
              'Problem: Where can you go to have a person assist you in '
              'mailing a package? A: USPS, B: Fedex, C: UPS',
 '2. response': "{'ans': 'A'}",
 '3. response_dict': {'ans': 'A'},
 '4. entropy': 0.094,
 '5. alternatives': [('A', 96.585, -0.03474311),
                     ('D', 3.384, -3.386195),
                     ('C', 0.023, -8.396432),
                     ('All', 0.004, -10.193547),
                     ('B', 0.004, -10.213874)]}


In [48]:
ans = get_answer(question=problems_common['ambiguous_03']['question'], prompt=multipleAnswerPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. There may be multiple answers. '
              'Provide a reasoning for why you made the choice. MAKE SURE to '
              'only provide one letter as an answer. Your final response '
              "should be a dictionary with key 'ans'.\n"
              'Problem: Where can you go to have a person assist you in '
              'mailing a package? A: USPS, B: Fedex, C: UPS',
 '2. response': "{'ans': 'A'}\n"
                '\n'
                'Reasoning: USPS is a government-run postal service where you '
                'can go to have a person assist you in mailing a package. '
                'Although Fedex and UPS also offer similar services, the '
                'question asks for only one answer.'

In [55]:
ans = get_answer(question=problems_common['ambiguous_03']['question'], prompt=multipleChoicesPROMPT, anskey='ans')
pprint.pprint(ans)

{'1. prompt': 'You will be given a commonsense reasoning problem. You will '
              'then be provided possible answer options labeled with letters '
              'following the question mark. Provide the letter assigned to the '
              'correct answer to the problem. There may be multiple answers. '
              'Provide a reasoning for why you made the choices. Give '
              'probabilities for the remaining probable answers. MAKE SURE to '
              'only provide one letter as an answer.   Your final response '
              "should be a dictionary with key 'ans'.\n"
              'Problem: Where can you go to have a person assist you in '
              'mailing a package? A: USPS, B: Fedex, C: UPS',
 '2. response': "{'ans': 'A', 'reasoning': 'All of the options provided, USPS, "
                'Fedex, and UPS, are places where you can go to have a person '
                'assist you in mailing a package. However, since the '
                'instruction i

In [56]:
%%capture cap --no-stderr

UsageError: %%capture is a cell magic, but the cell body is empty.
